# 00 — Verify Connection

Sanity-check Databricks Connect, confirm the DQX library is installed locally,
and create the catalog/schemas this demo writes into:

- `{UC_SCHEMA}_bronze`     — raw (dirty) synthetic pitches
- `{UC_SCHEMA}_silver`     — rows that survived DQX checks
- `{UC_SCHEMA}_quarantine` — rows DQX rejected (with `_errors` / `_warnings`)

Run this notebook **first** — every other notebook assumes these schemas exist.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "dqx_baseball")

BRONZE     = f"{UC_CATALOG}.{UC_SCHEMA}_bronze"
SILVER     = f"{UC_CATALOG}.{UC_SCHEMA}_silver"
QUARANTINE = f"{UC_CATALOG}.{UC_SCHEMA}_quarantine"

print(f"Bronze:     {BRONZE}")
print(f"Silver:     {SILVER}")
print(f"Quarantine: {QUARANTINE}")

Bronze:     alexander_booth.dqx_baseball_bronze
Silver:     alexander_booth.dqx_baseball_silver
Quarantine: alexander_booth.dqx_baseball_quarantine


In [2]:
# Confirm Spark is alive
spark.sql("SELECT current_user() AS user, current_version() AS dbr").show(truncate=False)

+------------------------------+-------------------------------------------------------------------------------------------------------------------+
|user                          |dbr                                                                                                                |
+------------------------------+-------------------------------------------------------------------------------------------------------------------+
|alexander.booth@databricks.com|{18.1.x-photon-scala2.13, NULL, 3d7f4c055f8cc3555c9c0ffee33652bd7045c2bd, 9f8c420c353f5b555aaa778cdc733e481bc0ede0}|
+------------------------------+-------------------------------------------------------------------------------------------------------------------+



In [3]:
# Confirm DQX is importable locally — apply_checks runs server-side via Databricks Connect,
# but we still need the engine + check_funcs in the local interpreter to define rules.
import databricks.labs.dqx as dqx
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx import check_funcs
from databricks.labs.dqx.rule import DQRowRule, DQDatasetRule

print(f"DQX version: {dqx.__version__}")

DQX version: 0.14.0


In [4]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {UC_CATALOG}")
for schema in (BRONZE, SILVER, QUARANTINE):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema} COMMENT 'DQX baseball demo'")
    print(f"  ready: {schema}")

  ready: alexander_booth.dqx_baseball_bronze


  ready: alexander_booth.dqx_baseball_silver


  ready: alexander_booth.dqx_baseball_quarantine


Connection verified. Continue with `01_generate_dirty_data`.